In [11]:
import pandas as pd
import sqlite3

In [6]:
df1_path = r"C:\Users\tonyw\Downloads\simplemaps_uscities_basicv1.78\uscities.csv" # us
df1 = pd.read_csv(df1_path)

df2_path = r"C:\Users\tonyw\Downloads\simplemaps_canadacities_basicv1.8\canadacities.csv" # canada
df2 = pd.read_csv(df2_path)

column_mapping = {
    'province_id': 'state_id',
    'province_name': 'state_name',
    'postal': 'zip'
}

df2_mapped = df2.rename(columns=column_mapping)


In [9]:
# Check if the column names are the same, and if not, align them
if not df1.columns.equals(df2_mapped.columns):
    # Add missing columns from df1 to df2
    missing_columns = df1.columns.difference(df2_mapped.columns)
    for column in missing_columns:
        df2_mapped[column] = None

    # Add missing columns from df2 to df1
    missing_columns = df2_mapped.columns.difference(df1.columns)
    for column in missing_columns:
        df1[column] = None

# Append df2 to df1
df1 = pd.concat([df1, df2_mapped], ignore_index=True)

In [12]:
# Create a connection to the SQLite database
conn = sqlite3.connect('ipg.sqlite')

# Insert df1 into the 'coordinates' table
df1.to_sql('coordinates', conn, if_exists='replace', index=False)

# Commit the changes and close the connection
conn.commit()
conn.close()


In [13]:
import pandas as pd
import sqlite3

# Connect to the SQLite database
conn = sqlite3.connect('ipg.sqlite')

# Read data from SQL tables into Pandas DataFrame
shipments_df = pd.read_sql_query('SELECT * FROM shipments', conn)
coordinates_df = pd.read_sql_query('SELECT * FROM coordinates', conn)

# Convert ship_to_city and state columns to lowercase for case-insensitive comparison
shipments_df['ship_to_city'] = shipments_df['ship_to_city'].str.lower()
shipments_df['state'] = shipments_df['state'].str.lower()
coordinates_df['city_ascii'] = coordinates_df['city_ascii'].str.lower()
coordinates_df['state_id'] = coordinates_df['state_id'].str.lower()

# Merge the DataFrames using case-insensitive comparison
merged_df = pd.merge(shipments_df, coordinates_df, 
                     left_on=['ship_to_city', 'state'], 
                     right_on=['city_ascii', 'state_id'], 
                     how='left')

# Select desired columns
result_df = merged_df[['bl_number', 'ship_to_city', 'state', 'ship_to_customer', 'lat', 'lng']]

# Print the result DataFrame
print(result_df)

     bl_number ship_to_city state                  ship_to_customer      lat  \
0      WZ1A912       irvine    ca                    MARUCHAN, INC.  33.6772   
1      WZ2A753      gardena    ca                      NISSIN FOODS  33.8943   
2      WZ3A004        chino    ca  WORLD CLASS DISTRIBUTION - CHINO  33.9836   
3      WZ3A004        chino    ca  WORLD CLASS DISTRIBUTION - CHINO  33.9836   
4      WZ3A060        lacey    wa      WORLD CLASS DISTRIBUTION -WA  47.0462   
...        ...          ...   ...                               ...      ...   
8002   2068803  white bluff    tn                   INTERSTATE PKG.  36.1005   
8003   2068803  white bluff    tn                   INTERSTATE PKG.  36.1005   
8004   2068803  white bluff    tn                   INTERSTATE PKG.  36.1005   
8005   2068805   cincinnati    oh                          CPG OHIO  39.1413   
8006   2068805   cincinnati    oh                          CPG OHIO  39.1413   

           lng  
0    -117.7738  
1    

In [4]:
import pandas as pd
from sklearn.cluster import DBSCAN
from geopy.distance import geodesic
import numpy as np
import matplotlib.pyplot as plt
import sqlite3
import folium
from folium.plugins import MarkerCluster



qry = ("SELECT s.site, s.product_group, s.bl_number, s.ship_to_customer, s.ship_to_city, s.state, "
            "SUM(s.pick_weight) AS net_pick_weight, SUM(s.number_of_pallet) AS total_number_of_pallet, "
            "c.lat, c.lng "
            "FROM shipments s "
            "LEFT JOIN coordinates c ON s.ship_to_city = upper(c.city_ascii) AND s.state = upper(c.state_id) "
            "WHERE s.site ='AMJK' AND s.product_group = 'SW'"
            "AND s.truck_appointment_date IS NULL AND s.product_code NOT LIKE 'INSERT%' AND s.bl_number NOT LIKE 'WZ%' "
            "GROUP BY s.bl_number "
            "ORDER BY s.state, s.ship_to_city ASC;")

# Step 1: Retrieve the latitude and longitude coordinates from SQLite database
conn = sqlite3.connect('ipg.sqlite')
data = pd.read_sql_query(qry, conn)
conn.close()

# Drop rows with NaN values
data.dropna(subset=['lat', 'lng'], inplace=True)

df = data.copy()


# # df = pd.read_csv(r"C:\Users\TonyW\Downloads\2024-04-12T16-54_export.csv")
# # data = {
# #     'lat': [46.67456497473562, 40.7128, 34.0522,   37.7749, 47.6062, 34.23665353995374, 46.22912475893864, 45.85682099975919, 40.71784300085435, 43.11890374783543, 43.089991997478315],
# #     'lng': [-122.65187649565887, -74.0060, -118.2437,    -122.4194, -122.3321,  -117.8707485081258, -122.88164287776588, -122.71493265741698, -74.05098331520506, -76.1502548312469, -87.9845301381357]
# # }

# # df = pd.DataFrame(data)

# # Assuming df is your DataFrame and it has columns 'lat' and 'lng'
df['coordinates'] = list(zip(df.lat, df.lng))

# # Convert latitude and longitude to radians
coords = np.radians(df[['lat', 'lng']].values)

# # Use DBSCAN to cluster the points based on distance
kms_per_radian = 6371.0088
epsilon = 95 / kms_per_radian  # 95 miles converted to radians
db = DBSCAN(eps=epsilon, min_samples=1, algorithm='ball_tree', metric='haversine').fit(coords)
cluster_labels = db.labels_

# Add the cluster labels to the DataFrame
df['cluster'] = cluster_labels

# # Post-process the clusters to split any clusters that have more than 3 points
clusters = df.groupby('cluster')['coordinates'].apply(list).to_dict()
new_clusters = []
for cluster in clusters.values():
    while len(cluster) > 3:
        new_clusters.append(cluster[:3])
        cluster = cluster[3:]
    new_clusters.append(cluster)

# # Now, 'new_clusters' is a list of clusters with each cluster containing at most 3 points
# # Print the clusters
for i, cluster in enumerate(new_clusters):
    print(f"Cluster {i+1}: {cluster}")

print (df)
print(f"Latitude range: {df['lat'].min()}, {df['lat'].max()}")
print(f"Longitude range: {df['lng'].min()}, {df['lng'].max()}")
# Initialize Folium map centered at the first data point
m = folium.Map(location=[df["lat"].mean(), df["lng"].mean()], zoom_start=4)
# Create MarkerCluster layer
marker_cluster = MarkerCluster().add_to(m)

# Add markers for each point, color-coded by their cluster
for cluster_label, cluster_points in enumerate(new_clusters):
    for point in cluster_points:
        folium.Marker(location=point[::-1], popup=f"Cluster: {cluster_label+1}").add_to(marker_cluster)

# Display the map inline
display (m)



Cluster 1: [(33.5279, -86.7971), (33.5279, -86.7971)]
Cluster 2: [(31.2336, -85.407)]
Cluster 3: [(34.6981, -86.6412)]
Cluster 4: [(32.3482, -86.2668), (32.3482, -86.2668)]
Cluster 5: [(33.3788, -88.0173)]
Cluster 6: [(34.4892, -93.0501)]
Cluster 7: [(35.8212, -90.6791), (35.8212, -90.6791)]
Cluster 8: [(31.3624, -110.9336)]
Cluster 9: [(33.5722, -112.0892)]
Cluster 10: [(49.1667, -123.1333)]
Cluster 11: [(33.9836, -117.6654), (33.8616, -117.5649), (33.8616, -117.5649)]
Cluster 12: [(34.0968, -117.4599), (34.0968, -117.4599), (34.0968, -117.4599)]
Cluster 13: [(33.8841, -117.9279), (33.8841, -117.9279), (34.0155, -118.1108)]
Cluster 14: [(33.9901, -118.0888), (33.9329, -118.0625), (33.9329, -118.0625)]
Cluster 15: [(33.4928, -117.1315), (34.0019, -118.2105), (34.0334, -117.8593)]
Cluster 16: [(34.0334, -117.8593), (34.0334, -117.8593), (34.0334, -117.8593)]
Cluster 17: [(41.9644, -121.9205)]
Cluster 18: [(36.6243, -119.6737), (36.783, -119.7939), (36.783, -119.7939)]
Cluster 19: [(37.6

In [18]:
# prep mass upload to sqlite

df_old = pd.read_csv(r"C:\Users\TonyW\Downloads\shipped_all_2.csv")
df_new = pd.read_csv(r"C:\Users\TonyW\Downloads\2024-04-12T16-54_export.csv")

# Define the mapping of old column names to new column names
# column_mapping = {
#     'Site':'',
#     'BL_Number':'',
#     'Truck_Appointment_Date':'',
#     'BL_weight':	Freight_Amount	Truck_Appt_Time	Pickup_Date	State	Ship_to_City'
# }

print(df_old.columns)
print(df_new.columns)

Index(['id', 'Site', 'BL_Number', 'Truck_Appointment_Date', 'BL_weight',
       'Freight_Amount', 'Truck_Appt_Time', 'Pickup_Date', 'State',
       'Ship_to_City', 'Ship_to_Customer', 'Order_Number', 'Order_Item', 'CSR',
       'Freight_Term', 'Require_Date', 'Schedule_Date', 'Unshipped_Weight',
       'Product_Code', 'Pick_Weight', 'Number_of_Pallet', 'Pickup_by',
       'Change_Date', 'Carrier_ID', 'Arrange_by', 'Unit_Freight',
       'Waybill_Number', 'Sales_Code', 'Transportation_Code',
       'transaction_type', 'Product_Group', 'rpt_run_date', 'rpt_run_time',
       'file_name', 'uploaded_date_time', 'file_size'],
      dtype='object')
Index(['Unnamed: 0', 'site', 'product_group', 'bl_number', 'ship_to_customer',
       'ship_to_city', 'state', 'net_pick_weight', 'total_number_of_pallet',
       'lat', 'lng'],
      dtype='object')


In [1]:
from sklearn.cluster import DBSCAN
from sklearn.metrics.pairwise import haversine_distances
from math import radians
import numpy as np
import folium

# Function to calculate distance between two points using Haversine formula
def haversine_distance(point1, point2):
    # convert decimal degrees to radians 
    lon1, lat1, lon2, lat2 = map(radians, [point1[0], point1[1], point2[0], point2[1]])
    # haversine formula 
    dlon = lon2 - lon1 
    dlat = lat2 - lat1 
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a)) 
    # Radius of earth in kilometers is 6371
    km = 6371 * c
    # Convert to miles
    miles = km * 0.621371
    return miles

# Function to get indices of nearest points within a distance threshold
def get_nearest_indices(points, distances, threshold):
    indices = []
    for i, dist in enumerate(distances):
        if dist <= threshold:
            indices.append(i)
    return indices

# Load your dataset (replace this with your own data loading code)
# Assume your data is in the format: [longitude, latitude]
data = [
    [-74.0059, 40.7128],  # Example point 1 (New York City)
    [-118.2437, 34.0522], # Example point 2 (Los Angeles)
    [-77.0369, 38.9072],  # Example point 3 (Washington, D.C.)
    [33.812172001224766, -117.91898493406354],
    # Add more points as needed
]

# Convert data to radians for distance calculation
data_radians = np.radians(data)

# Compute pairwise distances using Haversine formula
distances = haversine_distances(data_radians)

# Set maximum distance threshold in miles
max_distance_miles = 100
# Adjust epsilon based on maximum distance threshold
eps = max_distance_miles / 6371.  # Convert miles to kilometers
# DBSCAN clustering
dbscan = DBSCAN(eps=eps, min_samples=2, metric='precomputed')
clusters = dbscan.fit_predict(distances)

# Print clusters
print("Cluster labels:", clusters)

# Get unique cluster labels (excluding noise points)
unique_clusters = set(clusters) - {-1}  # -1 represents noise points
for cluster_label in unique_clusters:
    cluster_points_indices = np.where(clusters == cluster_label)[0]
    cluster_points = [data[i] for i in cluster_points_indices]
    print("Cluster {}: {}".format(cluster_label, cluster_points))

# Initialize Folium map centered at the first data point
center_lat, center_lon = data[0][1], data[0][0]
m = folium.Map(location=[center_lat, center_lon], zoom_start=5)

# Add markers for each point, color-coded by their cluster
for i, point in enumerate(data):
    cluster_label = clusters[i]
    color = 'gray' if cluster_label == -1 else 'blue'  # Use gray color for noise points
    folium.Marker(location=[point[1], point[0]], icon=folium.Icon(color=color), popup=f"Cluster: {cluster_label}").add_to(m)

# Display the map inline
display(m)

Cluster labels: [-1 -1 -1 -1]


In [5]:
# convert old table to sqlite
import pandas as pd
import sqlite3

df = pd.read_csv(r"C:\Users\TonyW\Downloads\shipped_all_2.csv") 
conn = sqlite3.connect('ipg.sqlite')
qry = f'select * from shipments'
new_df = pd.read_sql_query(qry, conn)


In [7]:
df.columns = df.columns.str.lower()
df.columns

Index(['id', 'site', 'bl_number', 'truck_appointment_date', 'bl_weight',
       'freight_amount', 'truck_appt_time', 'pickup_date', 'state',
       'ship_to_city', 'ship_to_customer', 'order_number', 'order_item', 'csr',
       'freight_term', 'require_date', 'schedule_date', 'unshipped_weight',
       'product_code', 'pick_weight', 'number_of_pallet', 'pickup_by',
       'change_date', 'carrier_id', 'arrange_by', 'unit_freight',
       'waybill_number', 'sales_code', 'transportation_code',
       'transaction_type', 'product_group', 'rpt_run_date', 'rpt_run_time',
       'file_name', 'uploaded_date_time', 'file_size'],
      dtype='object')

In [6]:
new_df.columns

Index(['id', 'site', 'bl_number', 'truck_appointment_date', 'bl_weight',
       'freight_amount', 'truck_appt_time', 'pickup_date', 'state',
       'ship_to_city', 'ship_to_customer', 'order_number', 'order_item', 'csr',
       'freight_term', 'require_date', 'schedule_date', 'unshipped_weight',
       'product_code', 'pick_weight', 'number_of_pallet', 'pickup_by',
       'change_date', 'carrier_id', 'arrange_by', 'unit_freight',
       'waybill_number', 'sales_code', 'transportation_code',
       'transaction_type', 'product_group', 'file_name', 'rpt_run_date',
       'rpt_run_time'],
      dtype='object')

In [ ]:
select bl_number, carrier_id, truck_appointment_date, sum(pick_weight), sum(number_of_pallet), ship_to_customer from shipments
where 
date(rpt_run_date)='2024-04-18'
and rpt_run_time=16
and site='AMJK'
and product_group='SW'
and truck_appointment_date= '2024-04-19'
AND carrier_id not in ('SAIA-IP', 'CWF-IP')
group by bl_number, carrier_id, truck_appointment_date 
ORDER by truck_appointment_date ASC


PINNACLE FILMS
INTEPLAST GROUP CORP.(AMTOPP ( CFP)
